# Ensemble 2 file kết quả retrieval bằng RRF (Reciprocal Rank Fusion)

## 1. Config — đường dẫn file input/output

In [ ]:
import json
from pathlib import Path

# ==== CHỈNH ĐƯỜNG DẪN Ở ĐÂY ====
FILE_1 = "DSC2026-IntelliJQK/outputs/submission_77.json"          # file kết quả thứ nhất
FILE_2 = "DSC2026-IntelliJQK/outputs/submission_7list33.json"          # file kết quả thứ hai
OUTPUT_FILE = "DSC2026-IntelliJQK/outputs/submission_77_7list33_60.json"  # file kết quả sau khi ensemble

TOP_K = 5      # số lượng doc cần lấy ra cho mỗi query
RRF_K = 60     # hằng số k trong công thức RRF (mặc định 60, giá trị phổ biến trong literature)


## 2. Load dữ liệu

In [2]:
def load_answers(path):
    """Load file json dạng {query_id: {"answer": [doc_id, ...]}} -> dict[str, list[str]]"""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return {qid: [str(d) for d in v["answer"]] for qid, v in data.items()}

answers_1 = load_answers(FILE_1)
answers_2 = load_answers(FILE_2)

print(f"File 1: {len(answers_1)} queries")
print(f"File 2: {len(answers_2)} queries")


File 1: 2080 queries
File 2: 2080 queries


In [3]:
def rrf_scores(rank_list, k=RRF_K):
    """Trả về dict {doc_id: score} theo công thức RRF cho 1 ranking (list doc_id theo thứ tự rank)."""
    return {doc: 1.0 / (k + rank) for rank, doc in enumerate(rank_list, start=1)}


def ensemble_query_rrf(list1, list2, top_k=TOP_K, k=RRF_K):
    """
    Ensemble 2 ranking (list doc_id) của cùng 1 query bằng RRF, ưu tiên doc chung.

    Quy tắc:
      1. Tính điểm RRF tổng (sum) cho mọi doc xuất hiện ở list1 hoặc list2.
      2. Lấy các doc XUẤT HIỆN Ở CẢ 2 LIST (giao nhau), sắp theo điểm RRF giảm dần.
         - Nếu số doc chung >= top_k -> lấy top_k doc chung có điểm cao nhất, DONE.
      3. Nếu chưa đủ top_k, lấy thêm các doc còn lại (chỉ ở 1 trong 2 list),
         sắp theo điểm RRF giảm dần, cho tới khi đủ top_k.
    """
    scores_1 = rrf_scores(list1, k)
    scores_2 = rrf_scores(list2, k)

    all_docs = set(list1) | set(list2)
    combined_scores = {
        doc: scores_1.get(doc, 0.0) + scores_2.get(doc, 0.0) for doc in all_docs
    }

    common_docs = set(list1) & set(list2)

    # Bước 2: doc chung, ưu tiên trước, sắp theo điểm RRF giảm dần
    common_sorted = sorted(common_docs, key=lambda d: combined_scores[d], reverse=True)

    final = common_sorted[:top_k]

    # Bước 3: nếu chưa đủ top_k thì bổ sung từ các doc còn lại theo điểm RRF
    if len(final) < top_k:
        remaining_candidates = [d for d in all_docs if d not in final]
        remaining_sorted = sorted(
            remaining_candidates, key=lambda d: combined_scores[d], reverse=True
        )
        need = top_k - len(final)
        final.extend(remaining_sorted[:need])

    return final, combined_scores


## 4. Chạy ensemble cho toàn bộ query

In [4]:
def ensemble_all(answers_1, answers_2, top_k=TOP_K, k=RRF_K):
    all_query_ids = set(answers_1.keys()) | set(answers_2.keys())
    result = {}

    for qid in all_query_ids:
        list1 = answers_1.get(qid, [])
        list2 = answers_2.get(qid, [])

        if list1 and list2:
            final, _ = ensemble_query_rrf(list1, list2, top_k=top_k, k=k)
        elif list1:
            # chỉ có ở file 1 -> lấy top_k đầu tiên của file 1
            final = list1[:top_k]
        else:
            # chỉ có ở file 2 -> lấy top_k đầu tiên của file 2
            final = list2[:top_k]

        result[qid] = {"answer": final}

    return result


ensembled = ensemble_all(answers_1, answers_2, top_k=TOP_K, k=RRF_K)
print(f"Đã ensemble {len(ensembled)} queries.")


Đã ensemble 2080 queries.


## 6. Lưu kết quả ra file

In [6]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(ensembled, f, ensure_ascii=False, indent=2)

print(f"Đã lưu kết quả ensemble vào: {OUTPUT_FILE}")


Đã lưu kết quả ensemble vào: /kaggle/working/submission_77_listgrid.json
